In [ ]:
import sys, types

SOURCE = "/kaggle/input/datasets/hervirakiza/source"
sys.path.insert(0, SOURCE)
src_pkg = types.ModuleType("src")
src_pkg.__path__ = [SOURCE]
src_pkg.__package__ = "src"
sys.modules["src"] = src_pkg

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold
from scipy.signal import savgol_filter
from scipy.ndimage import uniform_filter1d
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from data import load_well, list_wells
from features import engineer_features
from physics_prior import compute_physics_prior_for_well
import config

print("All imports OK")

## 2. Override Paths (Kaggle)

In [ ]:
import data, config
from pathlib import Path

COMP_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
data.TRAIN_DIR   = COMP_DIR / "train"
data.TEST_DIR    = COMP_DIR / "test"
config.TRAIN_DIR = data.TRAIN_DIR
config.TEST_DIR  = data.TEST_DIR

SAMPLE_SUB  = str(COMP_DIR / "sample_submission.csv")
OUTPUT_PATH = "/kaggle/working/submission.csv"

print(list(data.TRAIN_DIR.glob("*__horizontal_well.csv"))[:3])
print(f"SAMPLE_SUB  : {SAMPLE_SUB}")
print(f"OUTPUT_PATH : {OUTPUT_PATH}")

## 3. Feature Engineering

In [ ]:
# --- Basic feature cols (from config.py) ---
BASE_FEATURE_COLS = [
    "MD", "X", "Y", "Z", "GR",
    "dZ_dMD", "md_frac",
    "GR_roll_mean_5", "GR_roll_mean_25", "GR_roll_mean_50",
    "GR_roll_std_25", "GR_diff1",
    "GR_residual", "tw_GR_at_TVT",
    "TVT_ffill", "TVT_bfill", "TVT_linear_extrap",
    "md_since_last_known",
    "physics_prior_tvt",
    "physics_prior_corr",
]

# --- Rich feature cols (generated by add_rich_features below) ---
RICH_FEATURE_COLS = [
    "phys_roll_mean_3", "phys_roll_mean_5", "phys_roll_mean_11", "phys_roll_mean_21",
    "phys_roll_std_3", "phys_roll_std_5", "phys_roll_std_11", "phys_roll_std_21",
    "phys_lag_1", "phys_lag_2", "phys_lag_3", "phys_lag_5", "phys_lag_10",
    "phys_lead_1", "phys_lead_2", "phys_lead_3", "phys_lead_5", "phys_lead_10",
    "phys_diff1", "phys_diff2",
    "ffill_roll_mean_3", "ffill_roll_mean_7", "ffill_roll_mean_15",
    "ffill_roll_std_3", "ffill_roll_std_7", "ffill_roll_std_15",
    "ffill_lag_1", "ffill_lag_2", "ffill_lag_3", "ffill_lag_5",
    "ffill_lead_1", "ffill_lead_2", "ffill_lead_3", "ffill_lead_5",
    "bfill_roll_mean_3", "bfill_roll_mean_7",
    "bfill_lag_1", "bfill_lag_2", "bfill_lag_3",
    "bfill_lead_1", "bfill_lead_2", "bfill_lead_3",
    "prior_residual_phys_ffill", "prior_agreement",
    "prior_residual_phys_bfill",
    "ffill_bfill_mean", "ffill_bfill_diff",
    "phys_linext_diff",
    "MD_Z_ratio",
    "dZ_dMD_roll_mean_5", "dZ_dMD_roll_mean_11", "dZ_dMD_diff",
    "md_since_known_sq",
    "GR_tw_diff", "GR_x_phys",
]

print(f"BASE features: {len(BASE_FEATURE_COLS)}, RICH features: {len(RICH_FEATURE_COLS)}")

In [ ]:
def add_rich_features(hw: pd.DataFrame) -> pd.DataFrame:
    """Add rich features on top of base engineer_features + physics_prior."""
    df = hw.copy()
    c  = set(df.columns)

    # -- Rolling stats on physics_prior_tvt --
    if "physics_prior_tvt" in c:
        for w in [3, 5, 11, 21]:
            df[f"phys_roll_mean_{w}"] = df["physics_prior_tvt"].rolling(w, min_periods=1, center=True).mean()
            df[f"phys_roll_std_{w}"]  = df["physics_prior_tvt"].rolling(w, min_periods=1, center=True).std()
        for lag in [1, 2, 3, 5, 10]:
            df[f"phys_lag_{lag}"]  = df["physics_prior_tvt"].shift(lag)
            df[f"phys_lead_{lag}"] = df["physics_prior_tvt"].shift(-lag)
        df["phys_diff1"] = df["physics_prior_tvt"].diff()
        df["phys_diff2"] = df["physics_prior_tvt"].diff().diff()

    # -- Rolling stats on TVT_ffill --
    if "TVT_ffill" in c:
        for w in [3, 7, 15]:
            df[f"ffill_roll_mean_{w}"] = df["TVT_ffill"].rolling(w, min_periods=1, center=True).mean()
            df[f"ffill_roll_std_{w}"]  = df["TVT_ffill"].rolling(w, min_periods=1, center=True).std()
        for lag in [1, 2, 3, 5]:
            df[f"ffill_lag_{lag}"]  = df["TVT_ffill"].shift(lag)
            df[f"ffill_lead_{lag}"] = df["TVT_ffill"].shift(-lag)

    # -- Rolling stats on TVT_bfill --
    if "TVT_bfill" in c:
        for w in [3, 7]:
            df[f"bfill_roll_mean_{w}"] = df["TVT_bfill"].rolling(w, min_periods=1, center=True).mean()
        for lag in [1, 2, 3]:
            df[f"bfill_lag_{lag}"]  = df["TVT_bfill"].shift(lag)
            df[f"bfill_lead_{lag}"] = df["TVT_bfill"].shift(-lag)

    # -- Cross-prior features --
    if "physics_prior_tvt" in c and "TVT_ffill" in c:
        df["prior_residual_phys_ffill"] = df["physics_prior_tvt"] - df["TVT_ffill"]
        df["prior_agreement"]           = (df["physics_prior_tvt"] - df["TVT_ffill"]).abs()
    if "physics_prior_tvt" in c and "TVT_bfill" in c:
        df["prior_residual_phys_bfill"] = df["physics_prior_tvt"] - df["TVT_bfill"]
    if "TVT_ffill" in c and "TVT_bfill" in c:
        df["ffill_bfill_mean"] = (df["TVT_ffill"] + df["TVT_bfill"]) / 2
        df["ffill_bfill_diff"] = df["TVT_ffill"] - df["TVT_bfill"]
    if "physics_prior_tvt" in c and "TVT_linear_extrap" in c:
        df["phys_linext_diff"] = df["physics_prior_tvt"] - df["TVT_linear_extrap"]

    # -- Depth-based features --
    if "MD" in c and "Z" in c:
        df["MD_Z_ratio"] = df["MD"] / (df["Z"].abs() + 1e-6)
    if "dZ_dMD" in c:
        df["dZ_dMD_roll_mean_5"]  = df["dZ_dMD"].rolling(5, min_periods=1, center=True).mean()
        df["dZ_dMD_roll_mean_11"] = df["dZ_dMD"].rolling(11, min_periods=1, center=True).mean()
        df["dZ_dMD_diff"]         = df["dZ_dMD"].diff()
    if "md_since_last_known" in c:
        df["md_since_known_sq"] = df["md_since_last_known"] ** 2

    # -- GR interaction features --
    if "GR" in c and "tw_GR_at_TVT" in c:
        df["GR_tw_diff"] = df["GR"] - df["tw_GR_at_TVT"]
    if "GR_residual" in c and "physics_prior_tvt" in c:
        df["GR_x_phys"] = df["GR_residual"] * df["physics_prior_tvt"]

    # -- Ensemble prior (simple average of all available priors) --
    prior_cols = [col for col in ["physics_prior_tvt", "TVT_ffill", "TVT_bfill", "TVT_linear_extrap"] if col in c]
    if len(prior_cols) >= 2:
        df["prior_ensemble"] = df[prior_cols].mean(axis=1)

    return df

## 4. Build Training Data

In [ ]:
def build_training_data(wells, add_rich=True):
    """Build training dataset from all wells."""
    rows_list = []
    for wellname in tqdm(wells, desc="Processing train wells"):
        try:
            hw, tw = load_well(wellname, "train")
            hw = engineer_features(hw, tw)
            hw = compute_physics_prior_for_well(hw, tw, "train")
            if add_rich:
                hw = add_rich_features(hw)

            eval_mask = hw["TVT_input"].isna() & hw["TVT"].notna()
            if eval_mask.sum() == 0:
                continue

            available_base  = [c for c in BASE_FEATURE_COLS  if c in hw.columns]
            available_rich  = [c for c in RICH_FEATURE_COLS  if c in hw.columns]
            formation_cols  = [c for c in hw.columns if c.endswith("_dist")]
            extra_cols      = ["prior_ensemble"]
            all_feats_local = available_base + available_rich + formation_cols + extra_cols

            row = hw.loc[eval_mask, all_feats_local + ["TVT", "wellname"]]
            rows_list.append(row)
        except Exception as e:
            print(f"  Skipping {wellname}: {e}")
            continue

    return pd.concat(rows_list, ignore_index=True)


print("Building training dataset ...")
train_wells = list_wells("train")
print(f"Wells found : {len(train_wells)}")
train_df = build_training_data(train_wells)
print(f"Train eval samples : {len(train_df)}")

## 5. Prepare Features & Baseline RMSEs

In [ ]:
all_feats = [c for c in train_df.columns if c not in ("TVT", "wellname")]
print(f"Number of features : {len(all_feats)}")

X_train = train_df[all_feats].fillna(-9999)
y_train = train_df["TVT"]
groups  = train_df["wellname"]

print("\n=== Baseline RMSEs ===")
baseline_results = {}
for col, label in [
    ("TVT_ffill",        "FFILL baseline"),
    ("TVT_bfill",        "BFILL baseline"),
    ("TVT_linear_extrap","Linear extrap  "),
    ("physics_prior_tvt","Physics prior  "),
    ("prior_ensemble",   "Prior ensemble "),
]:
    if col in train_df.columns:
        rmse = np.sqrt(mean_squared_error(y_train, train_df[col].fillna(y_train.mean())))
        baseline_results[col] = rmse
        print(f"  {label} RMSE : {rmse:.4f}")

# Compute weighted prior ensemble (inverse RMSE weighting)
prior_cols_available = [k for k in ["physics_prior_tvt", "TVT_ffill", "TVT_bfill", "TVT_linear_extrap"] if k in baseline_results]
if len(prior_cols_available) >= 2:
    weights = {k: 1.0 / (baseline_results[k] + 1e-10) for k in prior_cols_available}
    total_w = sum(weights.values())
    for k in weights:
        weights[k] /= total_w
    weighted_ensemble = sum(train_df[k].fillna(y_train.mean()) * weights[k] for k in prior_cols_available)
    w_rmse = np.sqrt(mean_squared_error(y_train, weighted_ensemble))
    print(f"  Weighted ensemble RMSE : {w_rmse:.4f}")
    print(f"  Weights: { {k: f'{v:.3f}' for k, v in weights.items()} }")

## 6. LightGBM – 5-Fold GroupKFold (Round 1)

In [ ]:
LGB_PARAMS = {
    "objective":         "regression",
    "metric":            "rmse",
    "num_leaves":        127,
    "max_depth":         -1,
    "min_child_samples": 30,
    "learning_rate":     0.05,
    "feature_fraction":  0.75,
    "bagging_fraction":  0.75,
    "bagging_freq":      1,
    "reg_alpha":         0.05,
    "reg_lambda":        0.1,
    "n_estimators":      1500,
    "verbose":           -1,
    "n_jobs":            -1,
    "random_state":      42,
}

N_SPLITS  = 5
gkf       = GroupKFold(n_splits=N_SPLITS)
models    = []
oof_preds = np.zeros(len(X_train))
fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
    X_va, y_va = X_train.iloc[va_idx], y_train.iloc[va_idx]

    model = lgb.LGBMRegressor(**LGB_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )
    oof_preds[va_idx] = model.predict(X_va)
    fold_rmse = np.sqrt(mean_squared_error(y_va, oof_preds[va_idx]))
    fold_rmses.append(fold_rmse)
    models.append(model)
    print(f"  Fold {fold+1}/{N_SPLITS}  RMSE={fold_rmse:.4f}  best_iter={model.best_iteration_}")

oof_rmse_r1 = np.sqrt(mean_squared_error(y_train, oof_preds))
print(f"\nOOF RMSE (all folds) : {oof_rmse_r1:.4f}")
print(f"Mean fold RMSE        : {np.mean(fold_rmses):.4f}  \u00b1 {np.std(fold_rmses):.4f}")

## 7. Feature Importance & Selection

In [ ]:
imp = pd.DataFrame({
    "feature":    all_feats,
    "importance": np.mean([m.feature_importances_ for m in models], axis=0),
}).sort_values("importance", ascending=False)

print("Top 20 features:")
print(imp.head(20).to_string(index=False))
print(f"\nTotal features: {len(imp)}")

# Keep top-K features by cumulative importance
imp["cumul"] = imp["importance"].cumsum() / imp["importance"].sum()
KEEP_CUTOFF = 0.99
selected_feats = imp[imp["cumul"] <= KEEP_CUTOFF]["feature"].tolist()
if len(selected_feats) < 5:
    selected_feats = imp.head(5)["feature"].tolist()
removed = len(all_feats) - len(selected_feats)
print(f"Selected {len(selected_feats)} features (removed {removed} low-importance features)")
print(f"\nRemoved features: {imp[~imp['feature'].isin(selected_feats)]['feature'].tolist()}")

## 8. Retrain with Selected Features

In [ ]:
X_train_sel = X_train[selected_feats]

models_final = []
oof_preds_final = np.zeros(len(X_train_sel))
fold_rmses_final = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train_sel, y_train, groups)):
    X_tr, y_tr = X_train_sel.iloc[tr_idx], y_train.iloc[tr_idx]
    X_va, y_va = X_train_sel.iloc[va_idx], y_train.iloc[va_idx]

    model = lgb.LGBMRegressor(**LGB_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )
    oof_preds_final[va_idx] = model.predict(X_va)
    fold_rmse = np.sqrt(mean_squared_error(y_va, oof_preds_final[va_idx]))
    fold_rmses_final.append(fold_rmse)
    models_final.append(model)
    print(f"  Fold {fold+1}/{N_SPLITS}  RMSE={fold_rmse:.4f}  best_iter={model.best_iteration_}")

oof_rmse_r2 = np.sqrt(mean_squared_error(y_train, oof_preds_final))
print(f"\nOOF RMSE after feature selection: {oof_rmse_r2:.4f}")
print(f"Improvement: {oof_rmse_r1 - oof_rmse_r2:+.4f}")

## 9. Smoothing Post-Processing

In [ ]:
def smooth_oof_per_fold(oof_preds, y_train, groups, n_splits=5):
    """Apply Savitzky-Golay smoothing per fold."""
    smoothed = np.copy(oof_preds)
    gkf_local = GroupKFold(n_splits=n_splits)
    for i in range(n_splits):
        fold_mask = np.zeros(len(X_train_sel), dtype=bool)
        for j, (_, va_idx) in enumerate(gkf_local.split(X_train_sel, y_train, groups)):
            if j == i:
                fold_mask[va_idx] = True
        fp = oof_preds[fold_mask]
        if len(fp) >= 7:
            try:
                fp = savgol_filter(fp, window_length=7, polyorder=2)
            except Exception:
                fp = uniform_filter1d(fp, size=5)
        smoothed[fold_mask] = fp
    return smoothed


smoothed_final = smooth_oof_per_fold(oof_preds_final, y_train, groups)
smooth_rmse = np.sqrt(mean_squared_error(y_train, smoothed_final))
print(f"Smooth OOF RMSE : {smooth_rmse:.4f}")
print(f"Raw OOF RMSE    : {oof_rmse_r2:.4f}")

## 10. Build Test Submission

In [ ]:
def predict_ensemble(models, X):
    """Average predictions across fold models."""
    return np.mean([m.predict(X) for m in models], axis=0)


def smooth_predictions(preds, window: int = 7):
    """Savitzky-Golay smoothing."""
    if len(preds) >= window:
        try:
            return savgol_filter(preds, window_length=window, polyorder=2)
        except Exception:
            pass
    return uniform_filter1d(preds, size=min(5, len(preds)))


test_wells      = list_wells("test")
submission_rows = []

for wellname in tqdm(test_wells, desc="Predicting test wells"):
    try:
        hw, tw = load_well(wellname, "test")
        hw = engineer_features(hw, tw)
        hw = compute_physics_prior_for_well(hw, tw, "test")
        hw = add_rich_features(hw)

        eval_mask = hw["TVT_input"].isna()
        if eval_mask.sum() == 0:
            continue

        for col in selected_feats:
            if col not in hw.columns:
                hw[col] = -9999.0

        X_test = hw.loc[eval_mask, selected_feats].fillna(-9999)
        preds  = predict_ensemble(models_final, X_test)
        preds  = smooth_predictions(preds, window=7)

        # Clip to typewell range with 5% margin
        tw_data  = pd.read_csv(f"{config.TEST_DIR}/{wellname}__typewell.csv")
        tw_range = tw_data["TVT"].max() - tw_data["TVT"].min()
        margin   = 0.05 * tw_range
        preds    = np.clip(preds, tw_data["TVT"].min() - margin, tw_data["TVT"].max() + margin)

        row_indices = hw.loc[eval_mask].index
        for idx, pred in zip(row_indices, preds):
            submission_rows.append({"id": f"{wellname}_{idx}", "tvt": pred})
    except Exception as e:
        print(f"  Skipping test well {wellname}: {e}")
        continue

submission = pd.DataFrame(submission_rows)
print(f"Submission rows : {len(submission)}")

## 11. Align IDs & Save

In [ ]:
try:
    sample = pd.read_csv(SAMPLE_SUB)
    if len(submission) == len(sample):
        submission["id"] = sample["id"]
        print("ID aligned with sample submission")
    else:
        print(f"Warning: {len(submission)} rows vs {len(sample)} in sample -- skipping alignment")
except FileNotFoundError:
    print("sample_submission.csv not found -- saving with generated IDs")

submission.to_csv(OUTPUT_PATH, index=False)
print(f"Submission saved : {OUTPUT_PATH}  ({len(submission)} rows)")
print("Done!")